# MediSense-Cardio -- Arm B: the generator's own artifacts (wrinkles, creases, rotation, noise, crop, colour temperature)

**Why:** the researchers who built ECG-Image-Kit (and the PhysioNet Challenge) test digitisers on pages with realistic artifacts. Our main study uses six damage types
that we wrote ourselves. This small arm adds the tool's **own** artifact options, so the study also shows how the same digitiser behaves on those.

**What it does:** 30 recordings (a fixed rule, no result seen), each page re-drawn three ways with ECG-Image-Kit options:
**B1** wrinkles + creases, **B2** the tool's augmentation (random rotation up to 10 degrees, added noise, crop, colour temperature), **B3** both.
The clean page of each recording is the reference. Then the same digitiser, the same measurements (heart rate, intervals, aligned SNR), all 12 digitised leads saved to Drive.
About 30 x 4 = 120 images, roughly 40 minutes. Results: `results_armB.csv` on Drive. It is exploratory and changes no rule of the master plan.

**First run advice:** the augmentation (B2, B3) needs the library `imgaug`, which can fail on new NumPy versions. The notebook tests this on one record first and tells you clearly if B2/B3 cannot be made.
Run all, then run again if it stops.


## Step 0 -- Mount Google Drive and install base tools


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
STUDY_DIR = '/content/drive/MyDrive/MediSense-Cardio-Study'
os.makedirs(STUDY_DIR, exist_ok=True)
os.makedirs(f'{STUDY_DIR}/images_clean', exist_ok=True)
print(f"Study checkpoint directory: {STUDY_DIR}")


In [ ]:
!pip install -q wfdb neurokit2 opencv-python-headless pandas numpy scipy matplotlib
print("Base tools installed.")


## Step 1 -- The 30 recordings (embedded; every 3rd of the extension list)


In [ ]:
import pandas as pd, numpy as np, os, ast, ssl, urllib.request

SELECTED = [
    {'ecg_id': 115, 'filename_lr': 'records100/00000/00115_lr', 'sex_lbl': 'female', 'age': 18.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': True},
    {'ecg_id': 543, 'filename_lr': 'records100/00000/00543_lr', 'sex_lbl': 'female', 'age': 25.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 904, 'filename_lr': 'records100/00000/00904_lr', 'sex_lbl': 'male', 'age': 42.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': True},
    {'ecg_id': 1157, 'filename_lr': 'records100/01000/01157_lr', 'sex_lbl': 'male', 'age': 81.0, 'diagnostic_class': 'CD', 'is_abnormal': True, 'flagged': True},
    {'ecg_id': 2532, 'filename_lr': 'records100/02000/02532_lr', 'sex_lbl': 'male', 'age': 71.0, 'diagnostic_class': 'CD', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 3231, 'filename_lr': 'records100/03000/03231_lr', 'sex_lbl': 'female', 'age': 88.0, 'diagnostic_class': 'MI', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 4088, 'filename_lr': 'records100/04000/04088_lr', 'sex_lbl': 'female', 'age': 51.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 4695, 'filename_lr': 'records100/04000/04695_lr', 'sex_lbl': 'male', 'age': 82.0, 'diagnostic_class': 'STTC', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 5350, 'filename_lr': 'records100/05000/05350_lr', 'sex_lbl': 'male', 'age': 52.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 5843, 'filename_lr': 'records100/05000/05843_lr', 'sex_lbl': 'female', 'age': 39.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': True},
    {'ecg_id': 6046, 'filename_lr': 'records100/06000/06046_lr', 'sex_lbl': 'male', 'age': 62.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 7352, 'filename_lr': 'records100/07000/07352_lr', 'sex_lbl': 'male', 'age': 53.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': True},
    {'ecg_id': 7949, 'filename_lr': 'records100/07000/07949_lr', 'sex_lbl': 'female', 'age': 55.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 8233, 'filename_lr': 'records100/08000/08233_lr', 'sex_lbl': 'male', 'age': 61.0, 'diagnostic_class': 'CD', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 8733, 'filename_lr': 'records100/08000/08733_lr', 'sex_lbl': 'female', 'age': 70.0, 'diagnostic_class': 'STTC', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 9439, 'filename_lr': 'records100/09000/09439_lr', 'sex_lbl': 'female', 'age': 17.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 9951, 'filename_lr': 'records100/09000/09951_lr', 'sex_lbl': 'male', 'age': 47.0, 'diagnostic_class': 'STTC', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 10308, 'filename_lr': 'records100/10000/10308_lr', 'sex_lbl': 'male', 'age': 77.0, 'diagnostic_class': 'MI', 'is_abnormal': True, 'flagged': True},
    {'ecg_id': 10923, 'filename_lr': 'records100/10000/10923_lr', 'sex_lbl': 'male', 'age': 74.0, 'diagnostic_class': 'STTC', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 11894, 'filename_lr': 'records100/11000/11894_lr', 'sex_lbl': 'male', 'age': 39.0, 'diagnostic_class': 'HYP', 'is_abnormal': True, 'flagged': True},
    {'ecg_id': 12372, 'filename_lr': 'records100/12000/12372_lr', 'sex_lbl': 'female', 'age': 57.0, 'diagnostic_class': 'STTC', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 13767, 'filename_lr': 'records100/13000/13767_lr', 'sex_lbl': 'female', 'age': 57.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 14210, 'filename_lr': 'records100/14000/14210_lr', 'sex_lbl': 'male', 'age': 69.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 14505, 'filename_lr': 'records100/14000/14505_lr', 'sex_lbl': 'male', 'age': 70.0, 'diagnostic_class': 'MI', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 15111, 'filename_lr': 'records100/15000/15111_lr', 'sex_lbl': 'female', 'age': 25.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 15556, 'filename_lr': 'records100/15000/15556_lr', 'sex_lbl': 'female', 'age': 18.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': True},
    {'ecg_id': 16750, 'filename_lr': 'records100/16000/16750_lr', 'sex_lbl': 'female', 'age': 87.0, 'diagnostic_class': 'STTC', 'is_abnormal': True, 'flagged': False},
    {'ecg_id': 17060, 'filename_lr': 'records100/17000/17060_lr', 'sex_lbl': 'female', 'age': 300.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': True},
    {'ecg_id': 17762, 'filename_lr': 'records100/17000/17762_lr', 'sex_lbl': 'female', 'age': 56.0, 'diagnostic_class': 'NORM', 'is_abnormal': False, 'flagged': False},
    {'ecg_id': 18035, 'filename_lr': 'records100/18000/18035_lr', 'sex_lbl': 'female', 'age': 84.0, 'diagnostic_class': 'CD', 'is_abnormal': True, 'flagged': False}
]
sample = pd.DataFrame(SELECTED)
assert len(sample) == 30 and sample['ecg_id'].is_unique
print(f"Arm B: {len(sample)} recordings -- sex {sample['sex_lbl'].value_counts().to_dict()}, abnormal {int(sample['is_abnormal'].sum())}, flagged {int(sample['flagged'].sum())}.")


In [ ]:
BASE = 'https://physionet.org/files/ptb-xl/1.0.3/'
ctx = ssl.create_default_context()
with urllib.request.urlopen(BASE + 'ptbxl_database.csv', context=ctx) as r:
    with open('ptbxl_database_ext.csv', 'wb') as f:
        f.write(r.read())
dbx = pd.read_csv('ptbxl_database_ext.csv')[['ecg_id', 'sex', 'age', 'filename_hr', 'validated_by_human']]
sample = sample.merge(dbx[['ecg_id', 'sex', 'filename_hr', 'validated_by_human']], on='ecg_id', how='left')
assert sample['sex'].notna().all(), 'every selected record must be found in the PTB-XL label file'
# PTB-XL convention: 0 = male, 1 = female
assert (sample['sex'].map({0: 'male', 1: 'female'}) == sample['sex_lbl']).all(), 'embedded sex disagrees with PTB-XL'
assert sample['validated_by_human'].all(), 'every selected record must have human-validated labels'
print('Sex, ids and human-validation of all 100 recordings agree with the PTB-XL label file.')


## Step 2 -- Fetch the recordings (100 Hz and 500 Hz)


In [ ]:
import wfdb

# LOCAL PATCH: Colab's numpy removed the old np.fromstring binary-read path that
# wfdb's own code still calls internally.
import numpy as np
_original_fromstring = np.fromstring
def _patched_fromstring(string, dtype=float, count=-1, sep=''):
    if sep == '':
        return np.frombuffer(string, dtype=dtype, count=count)
    return _original_fromstring(string, dtype=dtype, count=count, sep=sep)
np.fromstring = _patched_fromstring

os.makedirs('/content/data/full', exist_ok=True)
os.makedirs('/content/data/full500', exist_ok=True)
record_ids = []
sex_by_rid = {}
for i, row in sample.iterrows():
    rel_dir = os.path.dirname(row['filename_lr'])
    rid = os.path.basename(row['filename_lr'])
    if not os.path.exists(f'/content/data/full/{rid}.hea'):
        record = wfdb.rdrecord(rid, pn_dir=f'ptb-xl/1.0.3/{rel_dir}')
        wfdb.wrsamp(rid, fs=record.fs, units=record.units, sig_name=record.sig_name,
                    p_signal=record.p_signal, write_dir='/content/data/full')
    if not os.path.exists(f'/content/data/full500/{rid}.hea'):
        rel_hr = os.path.dirname(row['filename_hr']); rid_hr = os.path.basename(row['filename_hr'])
        rec_hr = wfdb.rdrecord(rid_hr, pn_dir=f'ptb-xl/1.0.3/{rel_hr}')
        wfdb.wrsamp(rid, fs=rec_hr.fs, units=rec_hr.units, sig_name=rec_hr.sig_name,
                    p_signal=rec_hr.p_signal, write_dir='/content/data/full500')
    record_ids.append(rid)
    sex_by_rid[rid] = 'female' if row['sex'] == 1 else 'male'
    if (i + 1) % 100 == 0:
        print(f"  fetched {i + 1}/{len(sample)}...")
print(f"{len(record_ids)} recordings ready. Sex map covers {len(sex_by_rid)} records.")



## Step 3 -- Draw tool, then the three artifact versions of each page


In [ ]:
!git clone --depth 1 https://github.com/alphanumericslab/ecg-image-kit.git /content/ecg-image-kit
%cd /content/ecg-image-kit/codes/ecg-image-generator

# LOCAL PATCHES -- identical to the proven pilot notebook. Reused verbatim, not re-derived.
from pathlib import Path

HW_IMPORT = "from HandwrittenText.generate import get_handwritten"
p = Path('gen_ecg_image_from_data.py'); t = p.read_text()
if ("                " + HW_IMPORT) not in t:
    t = t.replace(HW_IMPORT + "\n", "")
    t = t.replace("            if(hw_text):", "            if(hw_text):\n                " + HW_IMPORT)
    p.write_text(t)

p = Path('helper_functions.py'); t = p.read_text()
if "from imgaug import augmenters as iaa\n" in t:
    p.write_text(t.replace("from imgaug import augmenters as iaa\n", ""))

p = Path('CreasesWrinkles/creases.py'); t = p.read_text()
if "import imgaug" in t:
    t = t.replace("import imgaug as ia\n", "").replace("from imgaug import augmenters as iaa\n", "")
    p.write_text(t)

p = Path('ImageAugmentation/augment.py'); t = p.read_text()
if "import imgaug" in t and "def get_augment" in t:
    lines = t.split('\n')
    top_imports = [l for l in lines if 'imgaug' in l and not l.strip().startswith('#')]
    for imp in top_imports:
        if lines.count(imp):
            lines.remove(imp)
    for i, l in enumerate(lines):
        if l.strip().startswith('def get_augment'):
            indent = '    '
            for imp in top_imports:
                lines.insert(i + 1, indent + imp.strip())
            break
    p.write_text('\n'.join(lines))
print("ecg-image-kit patched.")


In [ ]:
requirements_core = '''imageio
imutils
pillow
scikit-image
scikit-learn
scipy
matplotlib
joblib
numpy
pandas
wfdb
pyyaml
qrcode
'''
with open('requirements-core.txt', 'w') as f:
    f.write(requirements_core)
!pip install -q -r requirements-core.txt
print("Image-drawing tool ready.")


In [ ]:
import shutil

import glob

CLEAN_DIR_DRIVE = f'{STUDY_DIR}/images_clean'
os.makedirs('/content/images_clean', exist_ok=True)

already_rendered = {os.path.basename(p) for p in glob.glob(f'{CLEAN_DIR_DRIVE}/*.png')}
needs_render = [rid for rid in record_ids if f'{rid}-0.png' not in already_rendered]

if needs_render:
    print(f"Rendering {len(needs_render)} new record(s) (already have {len(already_rendered)} on Drive)...")
    os.makedirs('/content/data/to_render', exist_ok=True)
    for rid in needs_render:
        for ext in ('.hea', '.dat'):
            shutil.copy(f'/content/data/full/{rid}{ext}', f'/content/data/to_render/{rid}{ext}')
    !python gen_ecg_images_from_data_batch.py -i /content/data/to_render -o /content/images_clean_new --print_header
    for p in glob.glob('/content/images_clean_new/*.png'):
        shutil.copy(p, f'{CLEAN_DIR_DRIVE}/{os.path.basename(p)}')
    shutil.rmtree('/content/data/to_render')
    print(f"Rendered and saved {len(needs_render)} new clean image(s) to Drive.")
else:
    print("All clean images already on Drive -- nothing to render.")

# Sync only this session's scope (record_ids) from Drive to local disk -- not the whole Drive
# cache, which may hold images for a wider pool than the current scope is testing.
n_synced = 0
for rid in record_ids:
    drive_p = f'{CLEAN_DIR_DRIVE}/{rid}-0.png'
    local_p = f'/content/images_clean/{rid}-0.png'
    if os.path.exists(drive_p) and not os.path.exists(local_p):
        shutil.copy(drive_p, local_p)
        n_synced += 1
print(f"{n_synced} image(s) synced from Drive this session; {len(glob.glob('/content/images_clean/*.png'))} clean images available locally in total.")


In [ ]:
# imgaug is needed only for the augmentation (B2, B3). Old imgaug breaks on NumPy 2 (it uses np.sctypes), and the drawing tool runs as a
# SEPARATE program, so a fix inside this notebook is not enough. We write a tiny startup file (sitecustomize.py) that every Python process picks up.
import subprocess, sys, os
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imgaug'], capture_output=True, text=True)
print('pip install imgaug:', 'ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-600:])
SHIM_DIR = '/content/shim'
os.makedirs(SHIM_DIR, exist_ok=True)
with open(f'{SHIM_DIR}/sitecustomize.py', 'w') as f:
    f.write('''import numpy as _np
if not hasattr(_np, 'sctypes'):
    _np.sctypes = {'int': [_np.int8, _np.int16, _np.int32, _np.int64], 'uint': [_np.uint8, _np.uint16, _np.uint32, _np.uint64],
                   'float': [_np.float16, _np.float32, _np.float64], 'complex': [_np.complex64, _np.complex128],
                   'others': [bool, object, bytes, str]}
''')
TOOL_ENV = dict(os.environ, PYTHONPATH=SHIM_DIR + os.pathsep + os.environ.get('PYTHONPATH', ''))
# Second fix: the wrinkle code passes a NumPy number to cv2.subtract, which newer OpenCV versions reject. A plain Python float works everywhere.
_cp = '/content/ecg-image-kit/codes/ecg-image-generator/CreasesWrinkles/creases.py'
_t = open(_cp).read()
if 'shift = mean - 0.4' in _t:
    open(_cp, 'w').write(_t.replace('shift = mean - 0.4', 'shift = float(mean - 0.4)'))
    print('wrinkle code patched (float shift)')
t = subprocess.run([sys.executable, '-c', 'import imgaug; print(imgaug.__version__)'], capture_output=True, text=True, env=TOOL_ENV)
IMGAUG_OK = (t.returncode == 0)
print('imgaug usable by the drawing tool:', IMGAUG_OK, t.stdout.strip() if IMGAUG_OK else t.stderr[-400:])
if not IMGAUG_OK:
    print('B2 and B3 will be skipped; only B1 (wrinkles/creases) will run.')


In [ ]:
import shutil, glob, subprocess, os
GEN_DIR = '/content/ecg-image-kit/codes/ecg-image-generator'
COMMON = '--print_header'
CONDITIONS = {
    'B1_wrinkles': f'{COMMON} --wrinkles',
    # --store_config 1 is REQUIRED with --augment: the tool then writes the lead positions to a .json file that the augmentation reads.
    # Without it the augmentation crashes (TypeError: 'NoneType' object is not subscriptable) AFTER the plain page was already saved,
    # so a half-made page looks like a success. Found by local test 2026-11-05.
    'B2_augment':  f'{COMMON} --store_config 1 --augment -rot 10 -noise 25 -t 6500',
    'B3_both':     f'{COMMON} --store_config 1 --wrinkles --augment -rot 10 -noise 25 -t 6500',
}
if not IMGAUG_OK:
    CONDITIONS = {k: v for k, v in CONDITIONS.items() if '--augment' not in v}
    print('Without imgaug only these conditions can run:', list(CONDITIONS))

ARMB_DRIVE = f'{STUDY_DIR}/armB_images'
os.makedirs(ARMB_DRIVE, exist_ok=True)
os.makedirs('/content/armB_images', exist_ok=True)

def render_condition(name, flags, rids, tries=3):
    # One record per call, up to 3 tries: the drawing tool uses random choices and can crash on a page; one crash must not stop the others.
    src = f'/content/data/render_{name}'; out = f'/content/render_out_{name}'
    if os.path.exists(out): shutil.rmtree(out)
    os.makedirs(out)
    r = None
    for rid in rids:
        for attempt in range(tries):
            if os.path.exists(src): shutil.rmtree(src)
            os.makedirs(src)
            for ext in ('.hea', '.dat'):
                shutil.copy(f'/content/data/full/{rid}{ext}', f'{src}/{rid}{ext}')
            cmd = f'cd {GEN_DIR} && python gen_ecg_images_from_data_batch.py -i {src} -o {out} {flags}'
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=TOOL_ENV)
            # success = the tool finished without error AND the page exists (a crash after the plain page was saved leaves a half-made page behind)
            if r.returncode == 0 and glob.glob(f'{out}/{rid}-0.png'):
                break
        else:
            # all tries failed: delete the half-made page so it can never be counted as an artifact page
            for leftover in glob.glob(f'{out}/{rid}-0.*'):
                os.remove(leftover)
    pngs = sorted(glob.glob(f'{out}/*.png'))
    return r, pngs

# smoke test on one record first, so a broken option is found in a minute and not after the whole batch
test_rid = record_ids[0]
for name, flags in list(CONDITIONS.items()):
    r, pngs = render_condition(name, flags, [test_rid])
    ok = len(pngs) == 1
    print(f'smoke test {name}: {"OK" if ok else "FAILED"} ({len(pngs)} image)')
    if not ok:
        print((r.stdout + r.stderr)[-700:])
        CONDITIONS.pop(name)
print('Conditions that will run:', list(CONDITIONS))
assert CONDITIONS, 'no artifact condition could be made: paste the messages above into the chat'


In [ ]:
# Render every condition for all 30 recordings; copy to Drive and to the local working folder under the variant name.
render_report = {}
for name, flags in CONDITIONS.items():
    r, pngs = render_condition(name, flags, record_ids)
    got = {os.path.basename(p).replace('-0.png', ''): p for p in pngs}
    miss = [rid for rid in record_ids if rid not in got]
    render_report[name] = (len(got), len(miss))
    for rid, p in got.items():
        shutil.copy(p, f'/content/armB_images/{rid}-0__{name}.png')
        shutil.copy(p, f'{ARMB_DRIVE}/{rid}-0__{name}.png')
    print(f'{name}: {len(got)} of {len(record_ids)} images made, {len(miss)} missing {miss[:3]}')


## Step 4 -- Variant list: the clean page (reference) + the artifact versions


In [ ]:
import cv2
AXES = {}          # no own damage functions in this arm: images are used exactly as the generator drew them
LEVELS = [0]
variant_rows = []
for rid in record_ids:
    stem = f'{rid}-0'
    clean_path = f'/content/images_clean/{stem}.png'
    assert os.path.exists(clean_path), f'clean page missing: {clean_path}'
    variant_rows.append({'stem': stem, 'axis': 'clean', 'level': 0, 'variant_id': f'{stem}__clean0', 'source_path': clean_path})
    for name in CONDITIONS:
        p = f'/content/armB_images/{stem}__{name}.png'
        if os.path.exists(p):
            variant_rows.append({'stem': stem, 'axis': name, 'level': 0, 'variant_id': f'{stem}__{name}0', 'source_path': p})
all_variants = pd.DataFrame(variant_rows)
print(len(all_variants), 'variants:', all_variants['axis'].value_counts().to_dict())


## Step 5 -- Results checkpoint (separate file `results_armB.csv`)


In [ ]:
RESULTS_PATH = f'{STUDY_DIR}/results_armB.csv'

if os.path.exists(RESULTS_PATH):
    results_so_far = pd.read_csv(RESULTS_PATH)
    done_ids = set(results_so_far['variant_id'])
else:
    results_so_far = pd.DataFrame()
    done_ids = set()

remaining = all_variants[~all_variants['variant_id'].isin(done_ids)].reset_index(drop=True)
n_total, n_done, n_remaining = len(all_variants), len(done_ids), len(remaining)
print(f"{n_done}/{n_total} variants complete ({100*n_done/n_total:.1f}%). {n_remaining} remaining.")

if n_remaining == 0:
    print("\nALL VARIANTS COMPLETE -- nothing left to run. See the summary cells below.")




## Step 6 -- Set up the digitizer (once per session)


In [ ]:
import shutil
if os.path.exists('/content/Open-ECG-Digitizer'):
    shutil.rmtree('/content/Open-ECG-Digitizer')
!git clone https://github.com/Ahus-AIM/Open-ECG-Digitizer.git /content/Open-ECG-Digitizer
%cd /content/Open-ECG-Digitizer
!apt-get -qq install -y git-lfs
!git lfs install
!git lfs pull

path = 'src/utils.py'
with open(path) as f:
    content = f.read()
content = content.replace(
    "from ray.tune import Stopper",
    "try:\n    from ray.tune import Stopper\nexcept ImportError:\n    Stopper = object  # ray not needed for inference"
)
with open(path, 'w') as f:
    f.write(content)
print("Patched: ray import made optional")


In [ ]:
!pip install -q torch torchvision yacs scikit-image tqdm torch-tps
print("Digitizer tools ready.")


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

def write_digitizer_config(images_path, output_path):
    config_content = f'''MODEL:
  class_path: 'src.model.inference_wrapper.InferenceWrapper'
  KWARGS:
    config:
      SIGNAL_EXTRACTOR:
        class_path: 'src.model.signal_extractor.SignalExtractor'
        KWARGS: {{}}
      PERSPECTIVE_DETECTOR:
        class_path: 'src.model.perspective_detector.PerspectiveDetector'
        KWARGS:
          num_thetas: 250
      DEWARPER:
        class_path: 'src.model.dewarper.Dewarper'
        KWARGS:
          abs_peak_threshold: 0.1
      SEGMENTATION_MODEL:
        class_path: 'src.model.unet.UNet'
        weight_path: './weights/unet_weights_07072025.pt'
        KWARGS:
          num_in_channels: 3
          num_out_channels: 4
          dims: [32, 64, 128, 256, 320, 320, 320, 320]
          depth: 2
      CROPPER:
        class_path: 'src.model.cropper.Cropper'
        KWARGS:
          granularity: 80
          percentiles: [0.02, 0.98]
          alpha: 0.85
      PIXEL_SIZE_FINDER:
        class_path: 'src.model.pixel_size_finder.PixelSizeFinder'
        KWARGS:
          min_number_of_grid_lines: 30
          max_number_of_grid_lines: 70
          lower_grid_line_factor: 0.5
      LAYOUT_IDENTIFIER:
        class_path: 'src.model.lead_identifier.LeadIdentifier'
        config_path: 'src/config/lead_layouts_george-moody-2024.yml'
        unet_config_path: 'src/config/lead_name_unet.yml'
        unet_weight_path: './weights/lead_name_unet_weights_07072025.pt'
        KWARGS:
          debug: false
          device: '{device}'
          possibly_flipped: false
          target_num_samples: 10000
          required_valid_samples: 2
    device: '{device}'
    resample_size: 3000
    rotate_on_resample: true
    enable_timing: false
    apply_dewarping: false
DATA:
  images_path: '{images_path}'
  image_extensions: ['.png', '.jpg', '.jpeg', '.JPG']
  save_mode: 'all'
  layout_should_include_substring: false
  output_path: '{output_path}'
  clear_output_dir_if_exists: true
'''
    with open('/content/Open-ECG-Digitizer/batch_config.yml', 'w') as f:
        f.write(config_content)

print("Digitizer config helper ready.")


In [ ]:
import subprocess, json, datetime

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

prov = {
    'written_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'open_ecg_digitizer_commit': sh('git -C /content/Open-ECG-Digitizer rev-parse HEAD'),
    'ecg_image_kit_commit': sh('git -C /content/ecg-image-kit rev-parse HEAD'),
    'digitizer_weights': sorted(os.listdir('/content/Open-ECG-Digitizer/weights')) if os.path.isdir('/content/Open-ECG-Digitizer/weights') else [],
    'levels': LEVELS,
    'axes': list(AXES.keys()),
    'n_records': len(record_ids),
    'conditions': list(CONDITIONS),
    'imgaug_ok': IMGAUG_OK,
    'device': device,
}
with open(f'{STUDY_DIR}/armB_provenance.json', 'w') as f:
    json.dump(prov, f, indent=2)
print(json.dumps(prov, indent=2))



## Step 7 -- Digitise in checkpointed sub-batches (same loop as the main study)


In [ ]:
import time, neurokit2 as nk

SUBBATCH_SIZE = 20  # small so a disconnect loses little (CPU runs ~2 min/image)
SESSION_TIME_BUDGET_S = 10 * 3600  # leaves buffer inside Colab's connection limit
FS_CANONICAL = 1000.0

# Sex-specific QTc prolongation threshold, AHA/ACCF/HRS 2009 (docs/REFERENCES.md ref [52]):
# prolonged QTc is >=460ms in women, >450ms in men. Used only to classify QTc as
# in-range/out-of-range for the T3-style outcome computed in the analysis notebook --
# this notebook only saves the raw values and the classification flag, not an outcome verdict.
QTC_THRESHOLD_MS = {'female': 460.0, 'male': 450.0}

def extract_intervals(signal, fs, hr):
    """PR/QRS/QT/QTc in ms via NeuroKit2 delineation. NaN on any failure --
    unit-tested locally on synthetic ECG (scratchpad/test_intervals.py) before being
    embedded here verbatim."""
    out = {'pr_ms': float('nan'), 'qrs_ms': float('nan'), 'qt_ms': float('nan'), 'qtc_ms': float('nan')}
    try:
        cleaned = nk.ecg_clean(signal, sampling_rate=fs, method='neurokit')
        _, rpeaks = nk.ecg_peaks(cleaned, sampling_rate=fs)
        if rpeaks['ECG_R_Peaks'].sum() < 3:
            return out
        _, waves = nk.ecg_delineate(cleaned, rpeaks, sampling_rate=fs, method='dwt')
        p_on = np.array(waves.get('ECG_P_Onsets', []), dtype=float)
        r_on = np.array(waves.get('ECG_R_Onsets', []), dtype=float)
        r_off = np.array(waves.get('ECG_R_Offsets', []), dtype=float)
        t_off = np.array(waves.get('ECG_T_Offsets', []), dtype=float)

        def mean_diff_ms(a, b):
            n = min(len(a), len(b))
            if n == 0:
                return float('nan')
            d = (b[:n] - a[:n])
            d = d[~np.isnan(d)]
            return float(np.mean(d)) / fs * 1000.0 if len(d) else float('nan')

        out['pr_ms'] = mean_diff_ms(p_on, r_on)
        out['qrs_ms'] = mean_diff_ms(r_on, r_off)
        out['qt_ms'] = mean_diff_ms(r_on, t_off)
        if not np.isnan(out['qt_ms']) and hr and hr > 0 and not np.isnan(hr):
            rr_s = 60.0 / hr
            out['qtc_ms'] = out['qt_ms'] / np.sqrt(rr_s)
    except Exception:
        pass
    return out

os.makedirs(f'{STUDY_DIR}/armB_digitized_12lead', exist_ok=True)

def aligned_snr(gt500, dig_1000hz):
    """Challenge-style aligned SNR on one lead (lead II), in dB.
    dig_1000hz is the digitised signal at 1,000 Hz (mV); gt500 is the 500 Hz reference (mV).
    The digitised signal is brought to 500 Hz, both means are removed (vertical offset), and the best time shift within
    +-0.5 s is applied before SNR = 10*log10(sum(gt^2) / sum((dig - gt)^2)). Also returns the shift (ms) and Pearson r."""
    from scipy.signal import resample_poly, correlate, correlation_lags
    assert FS_CANONICAL == 1000.0
    d = resample_poly(np.asarray(dig_1000hz, dtype=float), 1, 2)
    g = np.asarray(gt500, dtype=float)
    n = min(len(g), len(d))
    if n < 500:
        return float('nan'), float('nan'), float('nan')
    g = g[:n] - np.mean(g[:n]); x = d[:n] - np.mean(d[:n])
    max_lag = 250  # 0.5 s at 500 Hz
    c = correlate(g, x, mode='full'); lags = correlation_lags(len(g), len(x), mode='full')
    keep = (lags >= -max_lag) & (lags <= max_lag)
    best = int(lags[keep][np.argmax(c[keep])])  # g[n + best] is aligned with x[n]
    if best >= 0:
        g2 = g[best:]; x2 = x[:len(g2)]
    else:
        x2 = x[-best:]; g2 = g[:len(x2)]
    g2 = g2 - np.mean(g2); x2 = x2 - np.mean(x2)
    noise = np.sum((x2 - g2) ** 2)
    if noise <= 0 or np.sum(g2 ** 2) <= 0:
        return float('nan'), float('nan'), float('nan')
    snr = float(10 * np.log10(np.sum(g2 ** 2) / noise))
    pcc = float(np.corrcoef(g2, x2)[0, 1])
    return snr, best / 500.0 * 1000.0, pcc

def compute_metrics(batch_df, digitized_dir):
    rows = []
    for _, v in batch_df.iterrows():
        gt_rid = v['stem'].split('-')[0]
        record = wfdb.rdrecord(f'/content/data/full/{gt_rid}')
        gt_signal = record.p_signal[:, record.sig_name.index('II')]
        fs_gt = record.fs
        sex = sex_by_rid.get(gt_rid, 'male')  # default only reached if lookup somehow missed -- see Step 1b assert

        csv_path = f'{digitized_dir}/{v["variant_id"]}_timeseries_canonical.csv'
        row = {'variant_id': v['variant_id'], 'record': gt_rid, 'axis': v['axis'], 'level': v['level'], 'sex': sex}

        gt_peaks, _ = nk.ecg_peaks(gt_signal, sampling_rate=fs_gt)
        gt_hr = np.nanmean(nk.ecg_rate(gt_peaks, sampling_rate=fs_gt, desired_length=len(gt_signal)))
        gt_iv = extract_intervals(gt_signal, fs_gt, gt_hr)
        row.update({
            'gt_hr': round(gt_hr, 1),
            'gt_pr_ms': round(gt_iv['pr_ms'], 1) if not np.isnan(gt_iv['pr_ms']) else None,
            'gt_qrs_ms': round(gt_iv['qrs_ms'], 1) if not np.isnan(gt_iv['qrs_ms']) else None,
            'gt_qt_ms': round(gt_iv['qt_ms'], 1) if not np.isnan(gt_iv['qt_ms']) else None,
            'gt_qtc_ms': round(gt_iv['qtc_ms'], 1) if not np.isnan(gt_iv['qtc_ms']) else None,
        })
        row['gt_qtc_abnormal'] = (
            bool(gt_iv['qtc_ms'] > QTC_THRESHOLD_MS[sex]) if not np.isnan(gt_iv['qtc_ms']) else None
        )

        # Reference values from the 500 Hz record (as in the journal analysis); NaN if it cannot be read.
        gt500_sig = None
        try:
            rec500 = wfdb.rdrecord(f'/content/data/full500/{gt_rid}')
            gt500_sig = rec500.p_signal[:, rec500.sig_name.index('II')]
            gt500_hr = np.nanmean(nk.ecg_rate(nk.ecg_peaks(gt500_sig, sampling_rate=rec500.fs)[0],
                                              sampling_rate=rec500.fs, desired_length=len(gt500_sig)))
            gt500_iv = extract_intervals(gt500_sig, rec500.fs, gt500_hr)
        except Exception:
            gt500_hr = float('nan')
            gt500_iv = {'pr_ms': float('nan'), 'qrs_ms': float('nan'), 'qt_ms': float('nan'), 'qtc_ms': float('nan')}
        row.update({
            'gt500_hr': round(gt500_hr, 1) if not np.isnan(gt500_hr) else None,
            'gt500_pr_ms': round(gt500_iv['pr_ms'], 1) if not np.isnan(gt500_iv['pr_ms']) else None,
            'gt500_qrs_ms': round(gt500_iv['qrs_ms'], 1) if not np.isnan(gt500_iv['qrs_ms']) else None,
            'gt500_qt_ms': round(gt500_iv['qt_ms'], 1) if not np.isnan(gt500_iv['qt_ms']) else None,
            'gt500_qtc_ms': round(gt500_iv['qtc_ms'], 1) if not np.isnan(gt500_iv['qtc_ms']) else None,
        })

        if not os.path.exists(csv_path):
            row.update({'dig_hr': None, 'diff_bpm': None, 'snr_aligned_db': None, 'snr_lag_ms': None, 'snr_pcc': None,
                        'dig_pr_ms': None, 'dig_qrs_ms': None, 'dig_qt_ms': None, 'dig_qtc_ms': None,
                        'dig_qtc_abnormal': None, 'diff_pr_ms': None, 'diff_qrs_ms': None,
                        'diff_qt_ms': None, 'diff_qtc_ms': None})
            rows.append(row)
            continue

        digitized = pd.read_csv(csv_path)
        dig_signal = digitized['II'].dropna().to_numpy() / 1000.0
        # keep ALL 12 leads exactly as the digitiser wrote them (microvolts, 1,000 Hz, NaN where a lead is missing)
        np.savez_compressed(f'{STUDY_DIR}/armB_digitized_12lead/{v["variant_id"]}.npz',
                            **{c: digitized[c].to_numpy(dtype='float32') for c in digitized.columns})

        try:
            dig_peaks, _ = nk.ecg_peaks(dig_signal, sampling_rate=FS_CANONICAL)
            dig_hr = np.nanmean(nk.ecg_rate(dig_peaks, sampling_rate=FS_CANONICAL, desired_length=len(dig_signal)))
        except Exception:
            dig_hr = float('nan')

        dig_iv = extract_intervals(dig_signal, FS_CANONICAL, dig_hr) if len(dig_signal) > 10 else {
            'pr_ms': float('nan'), 'qrs_ms': float('nan'), 'qt_ms': float('nan'), 'qtc_ms': float('nan')}

        diff = abs(gt_hr - dig_hr) if not np.isnan(dig_hr) else None

        def safe_diff(gt_v, dig_v):
            return abs(gt_v - dig_v) if (gt_v is not None and dig_v is not None
                                          and not np.isnan(gt_v) and not np.isnan(dig_v)) else None

        snr_aligned, snr_lag, snr_pcc = float('nan'), float('nan'), float('nan')
        if len(dig_signal) > 10 and gt500_sig is not None:
            snr_aligned, snr_lag, snr_pcc = aligned_snr(gt500_sig, dig_signal)

        row.update({
            'dig_hr': round(dig_hr, 1) if not np.isnan(dig_hr) else None,
            'diff_bpm': round(diff, 2) if diff is not None else None,
            'snr_aligned_db': round(snr_aligned, 2) if not np.isnan(snr_aligned) else None,
            'snr_lag_ms': round(snr_lag, 1) if not np.isnan(snr_lag) else None,
            'snr_pcc': round(snr_pcc, 3) if not np.isnan(snr_pcc) else None,
            'dig_pr_ms': round(dig_iv['pr_ms'], 1) if not np.isnan(dig_iv['pr_ms']) else None,
            'dig_qrs_ms': round(dig_iv['qrs_ms'], 1) if not np.isnan(dig_iv['qrs_ms']) else None,
            'dig_qt_ms': round(dig_iv['qt_ms'], 1) if not np.isnan(dig_iv['qt_ms']) else None,
            'dig_qtc_ms': round(dig_iv['qtc_ms'], 1) if not np.isnan(dig_iv['qtc_ms']) else None,
        })
        row['dig_qtc_abnormal'] = (
            bool(dig_iv['qtc_ms'] > QTC_THRESHOLD_MS[sex]) if not np.isnan(dig_iv['qtc_ms']) else None
        )
        row['diff_pr_ms'] = round(safe_diff(gt_iv['pr_ms'], dig_iv['pr_ms']), 1) if safe_diff(gt_iv['pr_ms'], dig_iv['pr_ms']) is not None else None
        row['diff_qrs_ms'] = round(safe_diff(gt_iv['qrs_ms'], dig_iv['qrs_ms']), 1) if safe_diff(gt_iv['qrs_ms'], dig_iv['qrs_ms']) is not None else None
        row['diff_qt_ms'] = round(safe_diff(gt_iv['qt_ms'], dig_iv['qt_ms']), 1) if safe_diff(gt_iv['qt_ms'], dig_iv['qt_ms']) is not None else None
        row['diff_qtc_ms'] = round(safe_diff(gt_iv['qtc_ms'], dig_iv['qtc_ms']), 1) if safe_diff(gt_iv['qtc_ms'], dig_iv['qtc_ms']) is not None else None
        rows.append(row)
    return pd.DataFrame(rows)

session_start = time.time()
n_processed_this_session = 0

while len(remaining) > 0 and (time.time() - session_start) < SESSION_TIME_BUDGET_S:
    batch = remaining.iloc[:SUBBATCH_SIZE].reset_index(drop=True)

    # Degrade: write this sub-batch's images.
    if os.path.exists('/content/images_degraded'):
        shutil.rmtree('/content/images_degraded')
    os.makedirs('/content/images_degraded', exist_ok=True)
    for _, v in batch.iterrows():
        img = cv2.imread(v['source_path'])
        out_img = AXES[v['axis']](img, v['level']) if v['axis'] in AXES else img   # arm B: images are used as drawn
        cv2.imwrite(f'/content/images_degraded/{v["variant_id"]}.png', out_img)

    # Digitize this sub-batch.
    if os.path.exists('/content/digitized_batch'):
        shutil.rmtree('/content/digitized_batch')
    write_digitizer_config('/content/images_degraded', '/content/digitized_batch')
    %cd /content/Open-ECG-Digitizer
    t0 = time.time()
    !python -m src.digitize --config /content/Open-ECG-Digitizer/batch_config.yml
    batch_elapsed = time.time() - t0

    # Metrics + checkpoint.
    batch_results = compute_metrics(batch, '/content/digitized_batch')
    results_so_far = pd.concat([results_so_far, batch_results], ignore_index=True)
    results_so_far = results_so_far.drop_duplicates(subset='variant_id', keep='last')
    # Safe save: write a temp file, swap it in atomically, keep a timestamped backup, force a
    # filesystem sync, then READ BACK from Drive and verify -- a lost save must show up now.
    tmp_path = RESULTS_PATH + '.tmp'
    results_so_far.to_csv(tmp_path, index=False)
    os.replace(tmp_path, RESULTS_PATH)
    shutil.copy(RESULTS_PATH, f'{STUDY_DIR}/results_armB_backup_{int(time.time())}.csv')
    os.sync()
    n_on_drive = len(pd.read_csv(RESULTS_PATH))
    assert n_on_drive == len(results_so_far), f'SAVE VERIFY FAILED: Drive has {n_on_drive}, memory has {len(results_so_far)}'
    print(f'  Verified on Drive: {n_on_drive} rows in results_armB.csv')

    remaining = remaining.iloc[SUBBATCH_SIZE:].reset_index(drop=True)
    n_processed_this_session += len(batch)
    rate = batch_elapsed / len(batch)
    elapsed_total = time.time() - session_start
    print(f"Checkpointed {len(batch)} variants ({rate:.1f}s/image). Session total: {n_processed_this_session} done, {len(remaining)} remaining. Session elapsed: {elapsed_total/60:.1f} min.")

n_done_total = len(results_so_far)
n_grand_total = len(all_variants)
print(f"\n=== Session summary ===")
print(f"Processed {n_processed_this_session} variants this session.")
print(f"Overall: {n_done_total}/{n_grand_total} complete ({100*n_done_total/n_grand_total:.1f}%).")
if len(remaining) == 0:
    print("ALL VARIANTS COMPLETE.")
else:
    est_remaining_hours = (rate * len(remaining)) / 3600 if n_processed_this_session > 0 else float('nan')
    print(f"{len(remaining)} variants remaining -- roughly {est_remaining_hours:.1f} more hours at this rate. Re-run this notebook (Runtime -> Run all) to continue.")




## Step 8 -- Overview


In [ ]:
results_so_far = pd.read_csv(RESULTS_PATH) if os.path.exists(RESULTS_PATH) else pd.DataFrame()
print(f'{len(results_so_far)}/{len(all_variants)} variants have results.')
if len(results_so_far) > 0:
    r = results_so_far.copy()
    r['no_output'] = r['dig_hr'].isna()
    r['wrong_hr_5bpm'] = r['diff_bpm'] > 5
    summ = r.groupby('axis').agg(n=('variant_id', 'size'), no_output=('no_output', 'sum'),
                                 wrong_hr_5bpm=('wrong_hr_5bpm', 'sum'), median_snr_db=('snr_aligned_db', 'median')).round(1)
    display(summ)
